# 03 - Radiomics feature estraction

## Import dependences

In [1]:
import os
from tqdm import tqdm
import pandas as pd
import numpy as np
import SimpleITK as sitk
sitk.ProcessObject.SetGlobalDefaultNumberOfThreads(1)
import cv2
import gc
import traceback

In [2]:
import radiomics
from radiomics import featureextractor
import logging
radiomics.setVerbosity(logging.ERROR)

## Transform images and mask to .nii.gz format

### Set paths

In [3]:
# original data paths
IMAGES_DIR = "F:/skin_cancer_data/data/images_selected"
MASKS_DIR = "F:/skin_cancer_data/data/masks_selected"

# transformed data paths
OUTPUT_IMAGES_DIR = "F:/skin_cancer_data/data/images_formated"

### Create output directories

In [4]:
output_types = ["images/blue/", "images/green/", "images/red/", "images/gray/"]

for output_type in output_types:
    path = os.path.join(OUTPUT_IMAGES_DIR, output_type)
    os.makedirs(path, exist_ok=True)

masks_path = os.path.join(OUTPUT_IMAGES_DIR, "masks/")
os.makedirs(masks_path, exist_ok=True)

### Transform and save the original images

In [5]:
images_names = os.listdir(IMAGES_DIR)
masks_names = os.listdir(MASKS_DIR)

In [ ]:
for image_name in tqdm(images_names, desc="Processing images"):
    # Define the output filename extension
    nii_name = image_name.replace(".jpg", ".nii.gz")
    
    # Define all target paths
    paths = {
        "blue": os.path.normpath(os.path.join(OUTPUT_IMAGES_DIR, "images", "blue", nii_name)),
        "green": os.path.normpath(os.path.join(OUTPUT_IMAGES_DIR, "images", "green", nii_name)),
        "red": os.path.normpath(os.path.join(OUTPUT_IMAGES_DIR, "images", "red", nii_name)),
        "gray": os.path.normpath(os.path.join(OUTPUT_IMAGES_DIR, "images", "gray", nii_name))
    }

    # Check: Only skip if ALL 4 files already exist
    if all(os.path.exists(p) for p in paths.values()):
        continue

    # Construct full input path
    image_path = os.path.join(IMAGES_DIR, image_name)
    
    # Read the image
    image_color = cv2.imread(image_path, cv2.IMREAD_COLOR)
    
    # Safety check: If image is corrupt or path is wrong, skip it
    if image_color is None:
        print(f"Warning: Could not read {image_name}")
        continue

    image_gray = cv2.cvtColor(image_color, cv2.COLOR_BGR2GRAY)

    # Separate channels (OpenCV is BGR)
    blue_channel = image_color[:, :, 0]
    green_channel = image_color[:, :, 1]
    red_channel = image_color[:, :, 2]

    # Convert to SimpleITK objects
    blue_channel_sitk = sitk.GetImageFromArray(blue_channel)
    green_channel_sitk = sitk.GetImageFromArray(green_channel)
    red_channel_sitk = sitk.GetImageFromArray(red_channel)
    gray_channel_sitk = sitk.GetImageFromArray(image_gray)
    
    # Save the files
    sitk.WriteImage(blue_channel_sitk, paths["blue"])
    sitk.WriteImage(green_channel_sitk, paths["green"])
    sitk.WriteImage(red_channel_sitk, paths["red"])
    sitk.WriteImage(gray_channel_sitk, paths["gray"])

### Transform and save the masks

In [ ]:
for image_name in tqdm(images_names, desc="Processing masks"):
    mask_path = os.path.join(MASKS_DIR, image_name)
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    mask_sitk = sitk.GetImageFromArray(mask)
    mask_path = os.path.join(masks_path, image_name.replace(".jpg", ".nii.gz"))
    sitk.WriteImage(mask_sitk, mask_path)


## Feature extractor

### Set output results path

In [6]:
OUTPUT_FEATURES_DIR = "F:/skin_cancer_data/data/features"
os.makedirs(OUTPUT_FEATURES_DIR, exist_ok=True)

### Select interest features (without 3d features)

In [7]:
extractor = featureextractor.RadiomicsFeatureExtractor()
extractor.disableAllFeatures()

features = ["firstorder", "glcm", "gldm", "glrlm", "glszm", "ngtdm", "shape2D"]
for feature in features:
    extractor.enableFeatureClassByName(feature)

### Create output dataframes

In [8]:
def to_scalar(value):
    if isinstance(value,(list,tuple,set)):
        return value[0]
    elif isinstance(value,dict):
        return value.values()[0]
    else:
        return value
        

### Extract features

In [ ]:
# --- Recomendado: poner ANTES de importar numpy / radiomics (si aplica) ---
import os
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")
os.environ.setdefault("ITK_GLOBAL_DEFAULT_NUMBER_OF_THREADS", "1")

import gc
import traceback
import pandas as pd
import SimpleITK as sitk
from tqdm import tqdm

# Evita crashes por multi-thread en ITK/SimpleITK
sitk.ProcessObject.SetGlobalDefaultNumberOfThreads(1)

out_csv = os.path.join(OUTPUT_FEATURES_DIR, "features_red_channel.csv")
log_path = os.path.join(OUTPUT_FEATURES_DIR, "errors_red_channel.log")
checkpoint_path = os.path.join(OUTPUT_FEATURES_DIR, "checkpoint_red_channel.txt")
os.makedirs(OUTPUT_FEATURES_DIR, exist_ok=True)

# Si el CSV ya existe, asumimos que ya tiene encabezado
header_written = os.path.exists(out_csv)

def to_nii_name(image_name: str) -> str:
    base, ext = os.path.splitext(image_name)
    ext = ext.lower()
    if ext in (".jpg", ".jpeg", ".png", ".tif", ".tiff", ".bmp", ".webp"):
        return base + ".nii.gz"
    # Si ya viene con otra extensión, igualmente lo convertimos
    return base + ".nii.gz"

for image_name in tqdm(images_names, desc="Processing images"):
    image_id = os.path.splitext(image_name)[0]

    # checkpoint inmediato (sirve si truena por segfault o se apaga)
    try:
        with open(checkpoint_path, "w", encoding="utf-8") as f:
            f.write(image_name + "\n")
    except Exception:
        # si por alguna razón falla el checkpoint, no detenemos el pipeline
        pass

    img = msk = raw = df_r = None

    try:
        nii_name = to_nii_name(image_name)

        # Construir paths
        img_path = os.path.join(OUTPUT_IMAGES_DIR, "images", "red", nii_name)
        msk_path = os.path.join(masks_path, nii_name)

        # Validar existencia de archivos (evita crashes por paths malos)
        if not os.path.exists(img_path):
            raise FileNotFoundError(f"Image not found: {img_path}")
        if not os.path.exists(msk_path):
            raise FileNotFoundError(f"Mask not found: {msk_path}")

        # Leer imagen y máscara
        img = sitk.ReadImage(img_path)
        msk = sitk.ReadImage(msk_path)

        # Validar tamaño
        if img.GetSize() != msk.GetSize():
            raise ValueError(f"Size mismatch img={img.GetSize()} mask={msk.GetSize()}")

        # Validar geometría (muy común que cause issues en radiomics)
        if (img.GetSpacing() != msk.GetSpacing() or
            img.GetOrigin()  != msk.GetOrigin()  or
            img.GetDirection()!= msk.GetDirection()):
            raise ValueError(
                "Geometry mismatch (spacing/origin/direction) "
                f"img(sp={img.GetSpacing()}, org={img.GetOrigin()}) "
                f"msk(sp={msk.GetSpacing()}, org={msk.GetOrigin()})"
            )

        # Máscara no vacía + binarizar (evita comportamientos raros)
        msk_view = sitk.GetArrayViewFromImage(msk)
        if msk_view.max() == 0:
            raise ValueError("Empty mask (all zeros)")

        msk = sitk.BinaryThreshold(msk, lowerThreshold=1, upperThreshold=1_000_000,
                                   insideValue=1, outsideValue=0)

        # Extraer features
        raw = extractor.execute(img, msk)

        # Convertir a una sola fila DataFrame (filtrando diagnostics_)
        row = {k: [to_scalar(v)] for k, v in raw.items() if "diagnostics_" not in k}
        df_r = pd.DataFrame(row, index=[image_id])

        # Append al CSV (sin concat en memoria)
        df_r.to_csv(
            out_csv,
            mode="a",
            header=(not header_written),
            index=True  # deja el image_id como índice en la primera columna
        )
        header_written = True

    except PermissionError as e:
        # Típico: el CSV está abierto en Excel u otro proceso
        with open(log_path, "a", encoding="utf-8") as f:
            f.write(f"\n---\nimage_id={image_id}\nPermissionError: {e}\n")
        raise  # detenemos para que cierres el archivo y reintentes

    except Exception:
        # Loggea el error y sigue con la siguiente imagen
        with open(log_path, "a", encoding="utf-8") as f:
            f.write(f"\n---\nimage_id={image_id}\nimage_name={image_name}\n")
            f.write(f"img_path={locals().get('img_path', 'NA')}\nmsk_path={locals().get('msk_path', 'NA')}\n")
            f.write(traceback.format_exc() + "\n")

    finally:
        # liberar memoria
        try:
            del img, msk, raw, df_r
        except Exception:
            pass
        gc.collect()


Processing images:   2%|▏         | 1836/74283 [22:49<11:59:35,  1.68it/s]

In [ ]:
for i, image_name in enumerate(tqdm(images_names, desc="Processing images")):
    image_id = image_name.split(".")[0]
    
    channel_green = sitk.ReadImage(os.path.join(OUTPUT_IMAGES_DIR, "images/green/", image_name.replace(".jpg", ".nii.gz")))
    mask = sitk.ReadImage(os.path.join(masks_path, image_name.replace(".jpg", ".nii.gz")))
    
    raw_results = extractor.execute(channel_green, mask)

    df_r = pd.DataFrame(
        {k: [to_scalar(v)] for k, v in raw_results.items() if "diagnostics_" not in k}, 
        index=[image_id]
    )

    if i == 0:
        df_r.to_csv(os.path.join(OUTPUT_FEATURES_DIR, "features_green_channel.csv"), mode='w', header=True)
    else:
        df_r.to_csv(os.path.join(OUTPUT_FEATURES_DIR, "features_green_channel.csv"), mode='a', header=False)

In [ ]:
for i, image_name in enumerate(tqdm(images_names, desc="Processing images")):
    image_id = image_name.split(".")[0]
    
    channel_blue = sitk.ReadImage(os.path.join(OUTPUT_IMAGES_DIR, "images/blue/", image_name.replace(".jpg", ".nii.gz")))
    mask = sitk.ReadImage(os.path.join(masks_path, image_name.replace(".jpg", ".nii.gz")))
    
    raw_results = extractor.execute(channel_blue, mask)

    df_r = pd.DataFrame(
        {k: [to_scalar(v)] for k, v in raw_results.items() if "diagnostics_" not in k}, 
        index=[image_id]
    )

    if i == 0:
        df_r.to_csv(os.path.join(OUTPUT_FEATURES_DIR, "features_blue_channel.csv"), mode='w', header=True)
    else:
        df_r.to_csv(os.path.join(OUTPUT_FEATURES_DIR, "features_blue_channel.csv"), mode='a', header=False)

In [ ]:
for i, image_name in enumerate(tqdm(images_names, desc="Processing images")):
    image_id = image_name.split(".")[0]
    
    channel_gray = sitk.ReadImage(os.path.join(OUTPUT_IMAGES_DIR, "images/gray/", image_name.replace(".jpg", ".nii.gz")))
    mask = sitk.ReadImage(os.path.join(masks_path, image_name.replace(".jpg", ".nii.gz")))
    
    raw_results = extractor.execute(channel_gray, mask)

    df_r = pd.DataFrame(
        {k: [to_scalar(v)] for k, v in raw_results.items() if "diagnostics_" not in k}, 
        index=[image_id]
    )

    if i == 0:
        df_r.to_csv(os.path.join(OUTPUT_FEATURES_DIR, "features_gray_channel.csv"), mode='w', header=True)
    else:
        df_r.to_csv(os.path.join(OUTPUT_FEATURES_DIR, "features_gray_channel.csv"), mode='a', header=False)

### Save results

In [ ]:
for image_name in tqdm(images_names, desc="Processing images"):
    image_id = image_name.split(".")[0]

    channel_blue = sitk.ReadImage(os.path.join(OUTPUT_IMAGES_DIR, "images/blue/", image_name.replace(".jpg", ".nii.gz")))
    channel_green = sitk.ReadImage(os.path.join(OUTPUT_IMAGES_DIR, "images/green/", image_name.replace(".jpg", ".nii.gz")))
    channel_red = sitk.ReadImage(os.path.join(OUTPUT_IMAGES_DIR, "images/red/", image_name.replace(".jpg", ".nii.gz")))
    channel_gray = sitk.ReadImage(os.path.join(OUTPUT_IMAGES_DIR, "images/gray/", image_name.replace(".jpg", ".nii.gz")))

    mask = sitk.ReadImage(os.path.join(masks_path, image_name.replace(".jpg", ".nii.gz")))
    
    results_blue_channel = extractor.execute(channel_blue, mask)
    results_blue_channel = extractor.execute(channel_green, mask)
    results_blue_channel = extractor.execute(channel_red, mask)
    results_blue_channel = extractor.execute(channel_gray, mask)

    df_r = pd.DataFrame({k: [to_scalar(v)] for k, v in results_blue_channel.items() if "diagnostics_" not in k}, index=[image_id])
    df_g = pd.DataFrame({k: [to_scalar(v)] for k, v in results_green_channel.items() if "diagnostics_" not in k}, index=[image_id])
    df_b = pd.DataFrame({k: [to_scalar(v)] for k, v in results_blue_channel.items() if "diagnostics_" not in k}, index=[image_id])
    df_gray = pd.DataFrame({k: [to_scalar(v)] for k, v in results_gray_channel.items() if "diagnostics_" not in k}, index=[image_id])

    results_red_channel = pd.concat([results_red_channel, df_r])
    results_green_channel = pd.concat([results_green_channel, df_g])
    results_blue_channel = pd.concat([results_blue_channel, df_b])
    results_gray_channel = pd.concat([results_gray_channel, df_gray])